In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 50)

In [0]:
# Read Employee_iteration CSV file from a Unity Catalog external volume
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/external-catalog/default/test-volume/Employee_Attrition.csv")

display(df)

In [0]:
# Filter employees with no attrition and low job satisfaction, then write to Delta
from pyspark.sql import functions as F

high_attrition_employees_df = (
    df.filter((F.col("Attrition") == "No") & (F.col("JobSatisfaction") < 3))
      .select(
          "EmployeeNumber",
          "Age",
          "Department",
          "JobRole",
          "JobLevel",
          "MonthlyIncome",
          "JobSatisfaction",
          "Attrition"
      )
)

high_attrition_employees_df.write.format("delta").mode("overwrite").saveAsTable(
    "`external-catalog`.default.high_attrition_employees"
)

display(high_attrition_employees_df)


In [0]:
# List Delta table version history
delta_table_name = "`external-catalog`.default.high_attrition_employees"

version_history_df = spark.sql(f"DESCRIBE HISTORY {delta_table_name}")

display(version_history_df.select("version", "timestamp", "userId", "userName", "operation", "operationParameters", "operationMetrics"))


In [0]:
# Read Delta table
df_delta = spark.read.format("delta").table("`external-catalog`.default.high_attrition_employees")

display(df_delta)

In [0]:
# Insert one dummy record into the Delta table
spark.sql("""
    INSERT INTO `external-catalog`.default.high_attrition_employees VALUES (
        9999,   -- EmployeeNumber
        35,     -- Age
        'Sales',           -- Department
        'Sales Executive', -- JobRole
        2,      -- JobLevel
        5000,   -- MonthlyIncome
        2,      -- JobSatisfaction
        'No'    -- Attrition
    )
""")

print("Dummy record inserted successfully.")

In [0]:
# List Delta table version history
delta_table_name = "`external-catalog`.default.high_attrition_employees"

version_history_df = spark.sql(f"DESCRIBE HISTORY {delta_table_name}")

display(version_history_df.select("version", "timestamp", "userId", "userName", "operation", "operationParameters", "operationMetrics"))

In [0]:
# Read Delta table
df_delta = spark.read.format("delta").table("`external-catalog`.default.high_attrition_employees")

display(df_delta)

In [0]:
# Read Delta table at a specific version using versionAsOf
df_v0 = (
    spark.read.format("delta")
    .option("versionAsOf", 3)  # Change version number as needed
    .table("`external-catalog`.default.high_attrition_employees")
)

display(df_v0)

In [0]:
# Read Delta table at a specific timestamp using timestampAsOf
df_ts = (
    spark.read.format("delta")
    .option("timestampAsOf", "2026-09-10T15:25:07.194+00:00")  # Change timestamp as needed
    .table("`external-catalog`.default.high_attrition_employees")
)

display(df_ts)

In [0]:
# Read Delta table at a specific timestamp using timestampAsOf
df_ts1 = (
    spark.read.format("delta")
    .option("timestampAsOf", "2026-09-10T15:34:10.957+00:00")  # Change timestamp as needed
    .table("`external-catalog`.default.high_attrition_employees")
)

display(df_ts1)

In [0]:
# Create a managed volume in the external-catalog
spark.sql("""
    CREATE VOLUME IF NOT EXISTS `external-catalog`.default.employee_transformed_data
""")

print("Managed volume 'employee_transformed_data' created successfully in 'external-catalog.default'.")

In [0]:
from pyspark.sql import functions as F

# --- Logical Transformations ---
df_transformed = (
    df
    # 1. Categorize age into bands
    .withColumn(
        "AgeGroup",
        F.when(F.col("Age") < 30, "Under 30")
         .when(F.col("Age").between(30, 45), "30-45")
         .otherwise("Over 45")
    )
    # 2. Flag high earners (monthly income above threshold)
    .withColumn(
        "HighEarner",
        F.when(F.col("MonthlyIncome") >= 8000, True).otherwise(False)
    )
    # 3. Derive tenure bucket from YearsAtCompany
    .withColumn(
        "TenureBucket",
        F.when(F.col("YearsAtCompany") <= 2, "0-2 yrs")
         .when(F.col("YearsAtCompany").between(3, 7), "3-7 yrs")
         .otherwise("8+ yrs")
    )
    # 4. Attrition risk score: low satisfaction + overtime = high risk
    .withColumn(
        "AttritionRiskScore",
        (5 - F.col("JobSatisfaction")) + F.when(F.col("OverTime") == "Yes", 3).otherwise(0)
    )
    # 5. Normalize MonthlyIncome to thousands
    .withColumn(
        "MonthlyIncomeK",
        F.round(F.col("MonthlyIncome") / 1000, 2)
    )
)

# --- Write to Unity Catalog Volume, partitioned by Department ---
output_path = "/Volumes/external-catalog/default/employee_transformed_data/transformed"

(
    df_transformed
    .write
    .format("parquet")
    .mode("overwrite")
    .partitionBy("Department")
    .save(output_path)
)

print(f"Transformed data written to volume at: {output_path}")
display(df_transformed)
